# OpenTelemetry Operator

Telemetrie ist der Prozess der automatischen Erfassung und Übertragung von Daten von entfernten oder verteilten Systemen, um deren Leistung und Status zu überwachen, zu messen und zu verfolgen. Telemetriedaten liefern Echtzeit-Einblicke in die Funktionsweise verschiedener Anwendungskomponenten. Sie stellen die Daten für Observability-Tools bereit, die Entwicklern und Systemadministratoren helfen, das System zu beobachten, Fehler zu beheben und es zu optimieren, ohne jede Komponente manuell überprüfen zu müssen.

**OpenTelemetry (OTel) ist ein Open-Source-Projekt**, das standardisierte Tools und APIs zum Generieren, Sammeln und Exportieren von Telemetriedaten wie Traces, Metriken und Logs bereitstellt. Ziel ist es, Entwicklern umfassende Einblicke in Anwendungen zu ermöglichen und sie bei der Überwachung, Fehlerbehebung und Optimierung von Softwaresystemen zu unterstützen.

Die Hauptziele von OpenTelemtry sind:

* Einheitliche Telemetrie : Kombiniert Tracing, Logging und Metriken in einem einzigen Framework, das die Korrelation aller Daten ermöglicht und einen offenen Standard für Telemetriedaten etabliert.
* Anbieterneutralität : Integration mit verschiedenen Backends zur Datenverarbeitung.
* Plattformübergreifend : Unterstützt verschiedene Sprachen (Java, Python, Go usw.) und Plattformen und ist daher vielseitig für unterschiedliche Entwicklungsumgebungen einsetzbar.


In [ ]:
%%bash
curl -sfL https://raw.githubusercontent.com/mc-b/lerncloud/main/services/cert-manager.sh | bash -

In [ ]:
%%bash
kubectl apply -f https://github.com/open-telemetry/opentelemetry-operator/releases/latest/download/opentelemetry-operator.yaml

In [ ]:
%%bash
kubectl get pods -n opentelemetry-operator-system

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: opentelemetry.io/v1beta1
kind: OpenTelemetryCollector
metadata:
  name: simplest
spec:
  config:
    receivers:
      otlp:
        protocols:
          grpc:
            endpoint: 0.0.0.0:4317
          http:
            endpoint: 0.0.0.0:4318
    processors:
      memory_limiter:
        check_interval: 1s
        limit_percentage: 75
        spike_limit_percentage: 15

    exporters:
      debug: {}

    service:
      pipelines:
        traces:
          receivers: [otlp]
          processors: [memory_limiter]
          exporters: [debug]
EOF

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: opentelemetry.io/v1beta1
kind: OpenTelemetryCollector
metadata:
  name: sidecar-for-my-app
spec:
  mode: sidecar
  config:
    receivers:
      jaeger:
        protocols:
          thrift_compact: {}
    processors:

    exporters:
      debug: {}

    service:
      pipelines:
        traces:
          receivers: [jaeger]
          exporters: [debug]
EOF

kubectl apply -f - <<EOF
apiVersion: v1
kind: Pod
metadata:
  name: myapp
  annotations:
    sidecar.opentelemetry.io/inject: "true"
spec:
  containers:
  - name: myapp
    image: jaegertracing/vertx-create-span:operator-e2e-tests
    ports:
      - containerPort: 8080
        protocol: TCP
EOF


In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: opentelemetry.io/v1alpha1
kind: Instrumentation
metadata:
  name: my-instrumentation
spec:
  exporter:
    endpoint: http://otel-collector:4317
  propagators:
    - tracecontext
    - baggage
    - b3
  sampler:
    type: parentbased_traceidratio
    argument: "0.25"
  python:
    env:
      # Required if endpoint is set to 4317.
      # Python autoinstrumentation uses http/proto by default
      # so data must be sent to 4318 instead of 4317.
      - name: OTEL_EXPORTER_OTLP_ENDPOINT
        value: http://otel-collector:4318
EOF